# Memory & Context Management for Multi-Turn Event Planning

This notebook adds a **memory layer** on top of the single-purpose agents from `Task.ipynb`, so a
user can refine an event brief (budget, headcount, location, dietary needs, ...) across many
conversational turns - and even across **separate sessions** - without re-stating everything each
time.

**Deliverables**

1. A `MemoryStore` class with three in-context (short-term) memory modes.
2. A context-compression function (`compress_turns`) that summarizes older turns so prompts stay
   bounded as a session grows.
3. A demonstration of preference recall across **two separate planning sessions** for the same
   user.

Everything below runs with no network access or API credentials: a `ScriptedLLMClient` (same
philosophy as `Task.ipynb`'s `FakeLLMClient`) stands in for a real LLM backend, and context
compression has a dependency-free deterministic fallback. Swap in `AzureOpenAIClient` for
production use - its contract is identical.

In [1]:
import json
import os
import sys
from collections import deque
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Type

from pydantic import BaseModel, Field

# Make the repo-root `init_creds.py` importable - only needed if/when `AzureOpenAIClient` below
# is actually instantiated.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


class LLMClient(Protocol):
    """Minimal interface every agent depends on (same contract as `Task.ipynb`)."""

    def generate(
        self,
        system_prompt: str,
        user_payload: Dict[str, Any],
        schema_name: str,
        json_schema: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a dict that should validate against `json_schema`."""
        ...


class AzureOpenAIClient:
    """Production LLM backend: Azure OpenAI chat completions with structured JSON output.

    Credentials are resolved lazily (via the repo's `init_creds.py` helpers) only when an
    instance is actually constructed, so importing/using this module never requires network
    access or secrets.
    """

    def __init__(self, model: str = "gpt-4o-mini", api_version: str = "2025-04-01-preview"):
        from openai import AzureOpenAI
        import init_creds as creds

        self._client = AzureOpenAI(
            api_key=creds.get_api_key(),
            azure_endpoint=creds.get_endpoint(),
            api_version=api_version,
        )
        self._model = model

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_payload)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": schema_name, "schema": json_schema, "strict": False},
            },
        )
        return json.loads(response.choices[0].message.content)


class ScriptedLLMClient:
    """Deterministic fake backend used throughout this notebook.

    Returns one canned response per call, popped off a scripted queue in order, so multi-turn
    demos can show a *different* structured output for each turn with no network access or API
    credentials - the same philosophy as `Task.ipynb`'s `FakeLLMClient`, extended to a sequence.
    """

    def __init__(self, responses: List[Dict[str, Any]]):
        self._responses = deque(responses)
        self.last_call: Optional[Dict[str, Any]] = None

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        self.last_call = {
            "system_prompt": system_prompt,
            "user_payload": user_payload,
            "schema_name": schema_name,
            "json_schema": json_schema,
        }
        if not self._responses:
            raise RuntimeError("ScriptedLLMClient ran out of canned responses")
        return self._responses.popleft()

## 1. Conversation turns and the compression function

A `Turn` is one message in the conversation, optionally tagged with the structured requirements it
contributed. `compress_turns` folds a list of turns into a single `ConversationSummary`: a short
narrative plus a `key_facts` dict where **later turns win** on conflicting values (e.g. a changed
budget supersedes the original one).

In [2]:
class Role(str, Enum):
    USER = "user"
    ASSISTANT = "assistant"
    SYSTEM = "system"


class Turn(BaseModel):
    """One message in a planning conversation, optionally tagged with the structured
    requirements it contributed (as produced by `RequirementExtractionAgent` below)."""

    turn_id: int
    role: Role
    content: str
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    extracted_requirements: Dict[str, Any] = Field(default_factory=dict)


class ConversationSummary(BaseModel):
    """Compressed stand-in for a block of older turns."""

    narrative_summary: str
    key_facts: Dict[str, Any] = Field(default_factory=dict)


def compress_turns(turns: List[Turn], llm: Optional[LLMClient] = None) -> ConversationSummary:
    """Summarize a list of older turns into a compact `ConversationSummary`.

    If an `llm` is supplied, delegate summarization to it (structured JSON output, same
    `LLMClient` contract as every agent). Otherwise fall back to a deterministic,
    dependency-free extractive summarizer, so compression can be demonstrated and tested
    with no network access or credentials.
    """
    if llm is not None:
        schema = ConversationSummary.model_json_schema()
        raw = llm.generate(
            system_prompt=(
                "You compress event-planning conversation history. Given a list of prior "
                "user/assistant turns, produce a concise narrative_summary (2-4 sentences) and a "
                "key_facts dict capturing concrete, still-relevant requirements (e.g. budget_usd, "
                "headcount, location, dietary_restrictions, event_date). Drop small talk and "
                "superseded facts - if a later turn changes an earlier value, keep only the "
                "latest. Respond ONLY with the structured JSON object defined by the schema."
            ),
            user_payload={"turns": [t.model_dump(mode="json") for t in turns]},
            schema_name="conversation_summary",
            json_schema=schema,
        )
        return ConversationSummary.model_validate(raw)

    # Deterministic fallback: merge every turn's structured facts (later turns win) and build a
    # short narrative from the raw turn content.
    merged_facts: Dict[str, Any] = {}
    for t in turns:
        merged_facts.update(t.extracted_requirements)

    narrative = " ".join(t.content for t in turns).strip()
    if len(narrative) > 280:
        narrative = narrative[:277].rsplit(" ", 1)[0] + "..."

    return ConversationSummary(
        narrative_summary=narrative or "(nothing to summarize)", key_facts=merged_facts
    )

## 2. `MemoryStore` - in-context memory with three modes

- `FULL` - keep every turn verbatim (no compression; bounded only by whatever the caller does
  with it).
- `BUFFER_WINDOW` - keep the last `window_size` raw turns; anything older is folded into a
  running `ConversationSummary`.
- `SUMMARY_ONLY` - collapse every turn into the running summary immediately; no raw turns persist
  between calls.

`known_requirements()` always returns the merged, de-duplicated view of every fact learned so
far, regardless of mode - that's what downstream planning agents should read. `export_preferences()`
/ `seed_from_preferences()` are the hooks used later for cross-session recall.

In [3]:
class MemoryMode(str, Enum):
    FULL = "full"
    BUFFER_WINDOW = "buffer_window"
    SUMMARY_ONLY = "summary_only"


class MemoryStore:
    """In-context (short-term) conversational memory for a single planning session.

    Once the raw turn count exceeds `window_size` (or immediately, in `SUMMARY_ONLY` mode),
    older turns are compressed via `compress_turns` and folded into a single running
    `ConversationSummary`, keeping prompts bounded regardless of how long the session runs.
    """

    def __init__(
        self,
        mode: MemoryMode = MemoryMode.BUFFER_WINDOW,
        window_size: int = 6,
        llm: Optional[LLMClient] = None,
    ):
        self.mode = mode
        self.window_size = window_size
        self._llm = llm
        self._turns: List[Turn] = []
        self._summary: Optional[ConversationSummary] = None
        self._next_id = 1

    def add_turn(
        self, role: Role, content: str, extracted_requirements: Optional[Dict[str, Any]] = None
    ) -> Turn:
        turn = Turn(
            turn_id=self._next_id,
            role=role,
            content=content,
            extracted_requirements=extracted_requirements or {},
        )
        self._next_id += 1
        self._turns.append(turn)
        self._compress_if_needed()
        return turn

    def _compress_if_needed(self) -> None:
        if self.mode == MemoryMode.FULL:
            return
        if self.mode == MemoryMode.SUMMARY_ONLY:
            self._summary = compress_turns(self._carry_forward() + self._turns, llm=self._llm)
            self._turns = []
            return
        if self.mode == MemoryMode.BUFFER_WINDOW and len(self._turns) > self.window_size:
            overflow = self._turns[: -self.window_size]
            self._turns = self._turns[-self.window_size :]
            self._summary = compress_turns(self._carry_forward() + overflow, llm=self._llm)

    def _carry_forward(self) -> List[Turn]:
        """Represent the existing running summary as a single pseudo-turn so it gets folded
        into the next compression pass instead of being discarded."""
        if self._summary is None:
            return []
        return [
            Turn(
                turn_id=0,
                role=Role.SYSTEM,
                content=self._summary.narrative_summary,
                extracted_requirements=self._summary.key_facts,
            )
        ]

    def known_requirements(self) -> Dict[str, Any]:
        """All structured facts learned so far - summarized facts merged with any still-raw
        turns' facts, with later turns taking precedence over earlier ones."""
        merged: Dict[str, Any] = dict(self._summary.key_facts) if self._summary else {}
        for t in self._turns:
            merged.update(t.extracted_requirements)
        return merged

    def context_for_llm(self) -> List[Dict[str, str]]:
        """Bounded message list to inject into the next LLM call - a running summary (if any)
        followed by whatever raw turns are still in the window."""
        messages: List[Dict[str, str]] = []
        if self._summary is not None:
            messages.append(
                {
                    "role": "system",
                    "content": (
                        f"Conversation summary so far: {self._summary.narrative_summary} "
                        f"Known requirements: {json.dumps(self._summary.key_facts)}"
                    ),
                }
            )
        messages.extend({"role": t.role.value, "content": t.content} for t in self._turns)
        return messages

    def stats(self) -> Dict[str, Any]:
        return {"raw_turns": len(self._turns), "has_summary": self._summary is not None}

    # --- cross-session persistence -----------------------------------------------------
    def export_preferences(self) -> Dict[str, Any]:
        """Serializable snapshot of everything learned - the long-term artifact a caller can
        persist (DB/file/cache) and later feed into a brand-new session's memory."""
        return self.known_requirements()

    def seed_from_preferences(self, preferences: Dict[str, Any], narrative: str = "") -> None:
        """Pre-load a fresh `MemoryStore` with preferences recalled from a previous session,
        without replaying that session's raw turns."""
        self._summary = ConversationSummary(
            narrative_summary=narrative or "Recalled preferences from a previous session.",
            key_facts=dict(preferences),
        )

## 3. Demonstrating context-window management

The same 12-turn conversation is replayed through all three modes. `FULL` grows without bound;
`BUFFER_WINDOW` stays at or below `window_size` raw turns; `SUMMARY_ONLY` never keeps any raw
turns at all. (This demo uses plain turns with no extracted facts, just to isolate the
turn-count/compaction behavior - the facts side is shown in the Session 1 demo further down.)

In [4]:
synthetic_conversation = [
    (Role.USER, "Hi, I'm planning a gala for our foundation."),
    (Role.ASSISTANT, "Great, happy to help - what's the headcount?"),
    (Role.USER, "Around 300 guests."),
    (Role.ASSISTANT, "Got it. Any budget in mind?"),
    (Role.USER, "We're targeting $250,000 total."),
    (Role.ASSISTANT, "Noted. Where should it be held?"),
    (Role.USER, "Somewhere in Miami, ideally waterfront."),
    (Role.ASSISTANT, "Any dietary restrictions to plan for?"),
    (Role.USER, "A handful of guests keep kosher."),
    (Role.ASSISTANT, "Understood, I'll flag that for catering."),
    (Role.USER, "Also we'll need a step-and-repeat and a livestream."),
    (Role.ASSISTANT, "Added to the AV requirements."),
]

for mode in (MemoryMode.FULL, MemoryMode.BUFFER_WINDOW, MemoryMode.SUMMARY_ONLY):
    store = MemoryStore(mode=mode, window_size=4)
    for role, content in synthetic_conversation:
        store.add_turn(role, content)
    print(f"{mode.value:>14}: {store.stats()}")

          full: {'raw_turns': 12, 'has_summary': False}
 buffer_window: {'raw_turns': 4, 'has_summary': True}
  summary_only: {'raw_turns': 0, 'has_summary': True}


## 4. A requirement-extraction agent (same `BaseAgent` pattern as `Task.ipynb`)

To attach structured `extracted_requirements` to each turn (what actually powers
`known_requirements()`), we reuse `Task.ipynb`'s `BaseAgent` scaffolding: a single-purpose agent
that reads the latest user message and extracts only the fields it explicitly mentions, leaving
everything else `null`.

In [5]:
class BaseAgent:
    """Shared scaffolding for single-purpose, structured-output agents (same contract as
    `Task.ipynb`'s `BaseAgent`)."""

    name: str
    system_prompt: str
    input_model: Type[BaseModel]
    output_model: Type[BaseModel]

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def run(self, request):
        if not isinstance(request, self.input_model):
            request = self.input_model.model_validate(request)
        payload = request.model_dump(mode="json")
        raw_output = self._llm.generate(
            system_prompt=self.system_prompt,
            user_payload=payload,
            schema_name=self.name,
            json_schema=self.output_model.model_json_schema(),
        )
        return self.output_model.model_validate(raw_output)


class ExtractionRequest(BaseModel):
    latest_message: str


class PartialEventRequirements(BaseModel):
    """Whatever subset of event requirements the user has mentioned so far - every field is
    optional because any single turn may only mention one or two new facts."""

    event_type: Optional[str] = None
    headcount: Optional[int] = None
    budget_usd: Optional[float] = None
    location: Optional[str] = None
    event_date: Optional[str] = None
    dietary_restrictions: Optional[List[str]] = None
    special_requirements: Optional[List[str]] = None


class RequirementExtractionAgent(BaseAgent):
    name = "requirement_extraction_agent"
    input_model = ExtractionRequest
    output_model = PartialEventRequirements
    system_prompt = (
        "You are a single-purpose requirement-extraction specialist for event planning chat.\n"
        "Given the latest user message, extract ONLY the event requirements it explicitly "
        "states or changes - leave every other field null.\n"
        "Respond ONLY with the structured JSON object defined by the provided schema."
    )

## 5. Session 1 - refining requirements across multiple turns

A user plans a product launch over five turns, mentioning one new fact at a time. The
`RequirementExtractionAgent` (backed by a `ScriptedLLMClient` with one canned extraction per
turn) attaches structured facts to each user turn, and `MemoryStore` (in `BUFFER_WINDOW` mode,
`window_size=4`) keeps the raw transcript bounded while `known_requirements()` keeps accumulating
the full picture. At the end of the session, the accumulated preferences are persisted to a
stand-in profile store, keyed by user id.

In [6]:
session1_extractor_llm = ScriptedLLMClient(
    responses=[
        {"event_type": "product_launch"},
        {"headcount": 200},
        {"budget_usd": 120000},
        {"location": "Chicago"},
        {"dietary_restrictions": ["vegan"]},
    ]
)
extraction_agent = RequirementExtractionAgent(llm=session1_extractor_llm)
session1_memory = MemoryStore(mode=MemoryMode.BUFFER_WINDOW, window_size=4)

session1_exchanges = [
    ("Hi, I need to plan a product launch event.", "Got it - a product launch. How many attendees?"),
    ("We're expecting about 200 attendees.", "Thanks. What's the budget?"),
    ("Budget is roughly $120,000 total.", "Noted. Where should it take place?"),
    ("It should be in Chicago, ideally downtown.", "Great choice. Any dietary needs?"),
    ("A few attendees are vegan, so we'll need vegan menu options.", "Understood, vegan options will be included."),
]

for user_msg, assistant_msg in session1_exchanges:
    extracted = extraction_agent.run({"latest_message": user_msg}).model_dump(exclude_none=True)
    session1_memory.add_turn(Role.USER, user_msg, extracted_requirements=extracted)
    session1_memory.add_turn(Role.ASSISTANT, assistant_msg)
    print(
        f"after '{user_msg[:45]}...' -> {session1_memory.stats()}\n"
        f"    known={session1_memory.known_requirements()}"
    )

PROFILE_DB: Dict[str, Dict[str, Any]] = {}  # stand-in for a real persistence layer (DB/file/cache)
PROFILE_DB["user_42"] = session1_memory.export_preferences()
print("\nPersisted cross-session profile for user_42:", PROFILE_DB["user_42"])

after 'Hi, I need to plan a product launch event....' -> {'raw_turns': 2, 'has_summary': False}
    known={'event_type': 'product_launch'}
after 'We're expecting about 200 attendees....' -> {'raw_turns': 4, 'has_summary': False}
    known={'event_type': 'product_launch', 'headcount': 200}
after 'Budget is roughly $120,000 total....' -> {'raw_turns': 4, 'has_summary': True}
    known={'event_type': 'product_launch', 'headcount': 200, 'budget_usd': 120000.0}
after 'It should be in Chicago, ideally downtown....' -> {'raw_turns': 4, 'has_summary': True}
    known={'event_type': 'product_launch', 'headcount': 200, 'budget_usd': 120000.0, 'location': 'Chicago'}
after 'A few attendees are vegan, so we'll need vega...' -> {'raw_turns': 4, 'has_summary': True}
    known={'event_type': 'product_launch', 'headcount': 200, 'budget_usd': 120000.0, 'location': 'Chicago', 'dietary_restrictions': ['vegan']}

Persisted cross-session profile for user_42: {'event_type': 'product_launch', 'headcount': 200

## 6. Session 2 - a brand-new conversation that recalls session 1's preferences

A new `MemoryStore` (fresh short-term memory - no raw turns from session 1) is seeded from the
persisted profile via `seed_from_preferences`. The user only mentions **one** change in this new
session; every other preference (event type, headcount, budget, dietary needs) is recalled
automatically instead of being asked for again.

In [7]:
session2_memory = MemoryStore(mode=MemoryMode.BUFFER_WINDOW, window_size=4)
session2_memory.seed_from_preferences(
    PROFILE_DB["user_42"],
    narrative="Recalled from a previous planning session for user_42's product launch.",
)
print("Session 2 starts already knowing:", session2_memory.known_requirements())

session2_extractor_llm = ScriptedLLMClient(responses=[{"location": "Denver"}])
session2_agent = RequirementExtractionAgent(llm=session2_extractor_llm)

user_msg = "Let's actually move the launch to Denver instead of Chicago."
extracted = session2_agent.run({"latest_message": user_msg}).model_dump(exclude_none=True)
session2_memory.add_turn(Role.USER, user_msg, extracted_requirements=extracted)
session2_memory.add_turn(Role.ASSISTANT, "Updated - Denver it is. Everything else stays the same.")

recalled = session2_memory.known_requirements()
print("Session 2 after one turn:", recalled)

assert recalled["location"] == "Denver", "new-session override should take effect"
assert recalled["budget_usd"] == 120000, "budget should be recalled from session 1"
assert recalled["headcount"] == 200, "headcount should be recalled from session 1"
assert recalled["event_type"] == "product_launch", "event type should be recalled from session 1"
assert "vegan" in recalled["dietary_restrictions"], "dietary needs should be recalled from session 1"
print("\nAll preference-recall checks passed.")

Session 2 starts already knowing: {'event_type': 'product_launch', 'headcount': 200, 'budget_usd': 120000.0, 'location': 'Chicago', 'dietary_restrictions': ['vegan']}
Session 2 after one turn: {'event_type': 'product_launch', 'headcount': 200, 'budget_usd': 120000.0, 'location': 'Denver', 'dietary_restrictions': ['vegan']}

All preference-recall checks passed.
